In [ ]:
# =============================================================================
# Real-ESRGAN Model - PSNR OPTIMIZED VERSION (BICUBIC DOWNSAMPLING)
# =============================================================================
# This version is optimized specifically for improving PSNR metrics.
# 
# Key Changes:
# 1. Bicubic downsampling instead of complex degradation (matches EDSR baseline)
# 2. Charbonnier loss (smooth L1) for better PSNR
# 3. Increased pixel loss weight (2.5x instead of 1.0x)
# 4. Reduced perceptual/adversarial loss weights (focus on pixel accuracy)
# 5. Same training epochs (30, aligned with baseline)
# 6. Model checkpointing based on PSNR performance
# 
# Expected Improvements:
# - Target: ~28-30+ dB PSNR (matching EDSR baseline performance)
# - Much higher pixel accuracy on bicubic test sets
# - Training and test distributions now match (bicubic → bicubic)
# 
# Training Strategy (aligned with baseline):
# - Single-stage training: 30 epochs (same as baseline)
# - Uses bicubic downsampling (same as EDSR baseline)
# - Uses Charbonnier loss (smooth L1) for better PSNR
# - Conservative increase in pixel loss weight (2.5x instead of 1.0x)
# - Same model structure (12 RRDB blocks)
# - Same GAN structure (discriminator and VGG still active)
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19
from torchvision.transforms import ToTensor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import sys
import random
import os
from pathlib import Path

plt.switch_backend('Agg')

# Dataset paths - relative to notebook location
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "datasets"
TRAIN_LR_PATH = DATASET_ROOT / "DIV2K_train_LR_bicubic" / "X4"
VALID_LR_PATH = DATASET_ROOT / "DIV2K_valid_LR_bicubic" / "X4"

SCALE_FACTOR = 4
BATCH_SIZE = 8  # Reduce to 6 or 4 if memory is tight (affects training speed)
LR_SIZE = 96
HR_SIZE = LR_SIZE * SCALE_FACTOR
TRAIN_SAMPLES = 800

# Memory optimization settings
USE_MIXED_PRECISION = True   # Use FP16 (half precision) - saves ~40-50% memory
ENABLE_GRADIENT_CHECKPOINTING = False  # Saves memory but slower (optional)
CLEAR_CACHE_FREQUENCY = 5   # Clear GPU cache every N batches (more frequent = lower memory)

# Output directory for results (separate from baseline)
OUTPUT_DIR = "results_psnr_improved"  # Directory for saving outputs
os.makedirs(OUTPUT_DIR, exist_ok=True)  # Create directory if it doesn't exist

# Degradation type: 'bicubic' (matches EDSR baseline and test set) or 'complex' (Real-ESRGAN style)
USE_BICUBIC_DEGRADATION = True  # Set to False to use complex degradation

# PSNR-focused hyperparameters (aligned with baseline structure)
EPOCHS = 30  # Keep same as baseline
USE_CHARBONNIER = True  # Use Charbonnier loss (smooth L1) - better for PSNR
USE_MSE_LOSS = False    # Use MSE instead of L1 (if Charbonnier is False)
LAMBDA_PIXEL = 4.0       # Increased to 4.0 to strongly emphasize pixel accuracy (maximize PSNR)
LAMBDA_PERCEP = 0.01     # Minimal perceptual loss (just for slight quality boost)
LAMBDA_ADV = 0.0         # Disable adversarial loss completely for maximum PSNR (GAN interferes with pixel accuracy)
GRADIENT_CLIP_VALUE = 1.0  # Less aggressive clipping for better convergence

# Model capacity (keep same as baseline or slightly increased)
NUM_RRDB_BLOCKS = 12    # Keep same as baseline (12 blocks)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")
print(f"Training Configuration: {EPOCHS} epochs (PSNR optimized)")
print("-" * 50)


PyTorch Version: 2.7.1+cu118
Device: cuda
--------------------------------------------------


In [2]:
def psnr(y_true, y_pred):
    mse = torch.mean((y_true - y_pred) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse + 1e-10))

def ssim(y_true, y_pred):
    """SSIM metric that handles both batched and single tensors."""
    # For batched tensors [B, C, H, W], compute per-sample SSIM then average
    # For single tensors [C, H, W], compute SSIM directly
    if len(y_true.shape) == 4:  # Batched: [B, C, H, W]
        mu1 = y_true.mean(dim=[1, 2, 3], keepdim=True)
        mu2 = y_pred.mean(dim=[1, 2, 3], keepdim=True)
        sigma1_sq = ((y_true - mu1) ** 2).mean(dim=[1, 2, 3], keepdim=True)
        sigma2_sq = ((y_pred - mu2) ** 2).mean(dim=[1, 2, 3], keepdim=True)
        sigma12 = ((y_true - mu1) * (y_pred - mu2)).mean(dim=[1, 2, 3], keepdim=True)
    else:  # Single sample: [C, H, W]
        mu1, mu2 = y_true.mean(), y_pred.mean()
        sigma1_sq = ((y_true - mu1) ** 2).mean()
        sigma2_sq = ((y_pred - mu2) ** 2).mean()
        sigma12 = ((y_true - mu1) * (y_pred - mu2)).mean()
    
    c1, c2 = 0.01**2, 0.03**2
    ssim_val = ((2*mu1*mu2 + c1) * (2*sigma12 + c2)) / ((mu1**2 + mu2**2 + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim_val.mean() if len(y_true.shape) == 4 else ssim_val


In [ ]:
# ----------------------------------------------------------------------
# 2. BICUBIC DOWNSAMPLING (Simple bicubic downsampling for training)
# ----------------------------------------------------------------------
# NOTE: Using simple bicubic downsampling to match EDSR baseline and test distribution

def generate_bicubic_lr(hr_img, scale=SCALE_FACTOR):
    """Generate LR image from HR using simple bicubic downsampling.
    
    Works on both CPU and GPU tensors. If hr_img is on GPU, computation happens on GPU
    for better utilization.
    """
    # Handle both batched [B, C, H, W] and unbatched [C, H, W] inputs
    if len(hr_img.shape) == 3:
        hr_img = hr_img.unsqueeze(0)
        squeeze_output = True
    else:
        squeeze_output = False
    
    h = hr_img.shape[-2] // scale
    w = hr_img.shape[-1] // scale
    # Use bicubic interpolation for downsampling (matches standard benchmarks like EDSR)
    # If hr_img is on GPU, this will run on GPU (better utilization)
    lr = F.interpolate(hr_img, size=(h, w), mode='bicubic', align_corners=False)
    
    if squeeze_output:
        lr = lr.squeeze(0)
    return lr


In [ ]:
# ----------------------------------------------------------------------
# 3. DATASET LOADING AND AUGMENTATION
# ----------------------------------------------------------------------

class DIV2KDataset(Dataset):
    """Custom Dataset for DIV2K with augmentation and bicubic downsampling."""
    def __init__(self, num_samples=TRAIN_SAMPLES, lr_size=LR_SIZE, hr_size=HR_SIZE, scale=SCALE_FACTOR, 
                 data_path=None, transform=None):
        self.num_samples = num_samples
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.scale = scale
        self.transform = transform
        self.images = []
        
        # Try to load DIV2K dataset from project folder
        data_path = data_path or TRAIN_LR_PATH
        try:
            print(f"Attempting to load DIV2K dataset from {data_path}...")
            if data_path.exists():
                # Load all PNG images from the folder
                image_files = sorted(list(data_path.glob("*.png")))[:num_samples]
                print(f"Found {len(image_files)} images in dataset folder.")
                
                if len(image_files) > 0:
                    for img_path in image_files:
                        img = Image.open(img_path).convert('RGB')
                        img_tensor = ToTensor()(img)  # Converts to [0,1] range and [C, H, W]
                        # Note: These are LR images. We'll use them and generate HR via upscaling for training
                        # The degradation pipeline will generate synthetic LR from HR patches
                        self.images.append(img_tensor)
                    print(f"Successfully loaded {len(self.images)} DIV2K images.")
                else:
                    raise FileNotFoundError(f"No PNG images found in {data_path}")
            else:
                raise FileNotFoundError(f"Dataset path does not exist: {data_path}")
        except Exception as e:
            print(f"DIV2K Loading failed: {e}. Using simulated data.")
            print(f"Creating synthetic DIV2K-like dataset...")
            self.images = [torch.rand(3, hr_size, hr_size) for _ in range(num_samples)]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]  # This could be LR or HR depending on what was loaded
        
        # If the loaded image is smaller than HR_SIZE, it's likely an LR image
        # Upscale it to HR_SIZE for training purposes
        if img.shape[1] < self.hr_size or img.shape[2] < self.hr_size:
            # Upscale LR to HR size using bilinear interpolation (CPU is fine here)
            img = F.interpolate(img.unsqueeze(0), size=(self.hr_size, self.hr_size), 
                               mode='bilinear', align_corners=False).squeeze(0)
        
        hr_img = img
        
        # Random crop to HR_SIZE
        if hr_img.shape[1] > self.hr_size or hr_img.shape[2] > self.hr_size:
            top = random.randint(0, hr_img.shape[1] - self.hr_size)
            left = random.randint(0, hr_img.shape[2] - self.hr_size)
            # Align to scale factor
            top = (top // self.scale) * self.scale
            left = (left // self.scale) * self.scale
            hr_img = hr_img[:, top:top+self.hr_size, left:left+self.hr_size]
        
        # Return HR only - do bicubic downsampling on GPU for better utilization
        # This moves computation from CPU (blocking) to GPU (parallel)
        
        # Apply augmentation (CPU is fine for simple ops)
        if random.random() < 0.5:
            hr_img = torch.flip(hr_img, dims=[2])
        
        if random.random() < 0.5:
            k = random.randint(0, 3)
            hr_img = torch.rot90(hr_img, k, dims=[1, 2])
        
        # Return HR only - LR will be generated on GPU
        return hr_img

def load_or_simulate_dataset(num_samples, batch_size, lr_shape, hr_shape):
    """
    Loads the real DIV2K dataset and applies preprocessing and augmentation.
    """
    try:
        print("Attempting to load real DIV2K dataset...")
        
        # In practice, you would download and load DIV2K images from disk
        # For now, we simulate with synthetic data
        dataset = DIV2KDataset(num_samples=num_samples, lr_size=lr_shape[0], hr_size=hr_shape[0])
        
        # Apply preprocessing, shuffling, and batching
        # Windows requires num_workers=0 due to multiprocessing limitations
        # The key optimization is moving LR generation to GPU (done in training loop)
        num_workers = 0  # Must be 0 on Windows, but GPU bicubic generation still improves utilization
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=num_workers, pin_memory=True)
        
        print(f"TFDS DIV2K dataset processing configured with BATCH_SIZE={batch_size}.")
        return dataloader
    
    except Exception as e:
        print(f"TFDS Loading failed: {e}. Falling back to simulation.")
        print(f"Creating SIMULATED DIV2K dataset: {num_samples} samples, batch size {batch_size}")
        dataset = DIV2KDataset(num_samples=num_samples, lr_size=lr_shape[0], hr_size=hr_shape[0])
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=0, pin_memory=True)
        return dataloader


In [ ]:
# ----------------------------------------------------------------------
# 3.5. TEST DATASET (BSD100) - Load HR and LR images from dataset
# ----------------------------------------------------------------------

class BSD100Dataset(Dataset):
    """Custom Dataset for BSD100 test set with HR and LR pairs."""
    def __init__(self, hr_path=None, lr_path=None, scale=SCALE_FACTOR):
        self.scale = scale
        self.lr_images = []
        self.hr_images = []
        
        # Default paths for BSD100 dataset
        if hr_path is None:
            hr_path = DATASET_ROOT / "bsd100" / f"bicubic_{scale}x" / "val" / "HR"
        if lr_path is None:
            lr_path = DATASET_ROOT / "bsd100" / f"bicubic_{scale}x" / "val" / "LR"
        
        try:
            print(f"\nAttempting to load BSD100 Test Set...")
            print(f"  HR path: {hr_path}")
            print(f"  LR path: {lr_path}")
            
            if not hr_path.exists():
                raise FileNotFoundError(f"HR path does not exist: {hr_path}")
            if not lr_path.exists():
                raise FileNotFoundError(f"LR path does not exist: {lr_path}")
            
            # Load HR images
            hr_files = sorted(list(hr_path.glob("*.png")))
            lr_files = sorted(list(lr_path.glob("*.png")))
            
            print(f"Found {len(hr_files)} HR images and {len(lr_files)} LR images.")
            
            if len(hr_files) == 0:
                raise FileNotFoundError(f"No HR images found in {hr_path}")
            if len(lr_files) == 0:
                raise FileNotFoundError(f"No LR images found in {lr_path}")
            
            # Match HR and LR images by filename
            hr_dict = {f.stem: f for f in hr_files}
            lr_dict = {f.stem: f for f in lr_files}
            
            # Find matching pairs
            matched_count = 0
            for hr_name in sorted(hr_dict.keys()):
                if hr_name in lr_dict:
                    # Load HR image
                    hr_img = Image.open(hr_dict[hr_name]).convert('RGB')
                    hr_tensor = ToTensor()(hr_img)  # [C, H, W], [0,1]
                    
                    # Load LR image
                    lr_img = Image.open(lr_dict[hr_name]).convert('RGB')
                    lr_tensor = ToTensor()(lr_img)  # [C, H, W], [0,1]
                    
                    self.hr_images.append(hr_tensor)
                    self.lr_images.append(lr_tensor)
                    matched_count += 1
            
            print(f"Successfully loaded {matched_count} matched HR-LR pairs from BSD100.")
            
            if matched_count == 0:
                raise ValueError("No matching HR-LR pairs found!")
                
        except Exception as e:
            print(f"BSD100 Loading failed: {e}")
            print("Creating synthetic BSD100-like test dataset...")
            # Fallback: create synthetic pairs
            sizes = [(256, 256), (384, 384), (512, 384), (384, 512), (512, 512)]
            for h, w in sizes * 4:  # 20 samples
                hr_img = torch.rand(3, h, w)
                lr_h, lr_w = h // scale, w // scale
                lr_img = F.interpolate(hr_img.unsqueeze(0), size=(lr_h, lr_w), 
                                      mode='bicubic', align_corners=False).squeeze(0)
                self.hr_images.append(hr_img)
                self.lr_images.append(lr_img)
            print(f"Created {len(self.hr_images)} synthetic test pairs.")
    
    def __len__(self):
        return len(self.hr_images)
    
    def __getitem__(self, idx):
        return self.lr_images[idx], self.hr_images[idx]

def load_test_dataset(scale=SCALE_FACTOR):
    """
    Loads the BSD100 dataset for testing with HR and LR image pairs.
    """
    try:
        print("\nAttempting to load BSD100 (B100) Test Set...")
        dataset = BSD100Dataset(scale=scale)
        # Note: num_workers=0 to avoid multiprocessing issues on Windows
        dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
        print("BSD100 Test Set configured for evaluation.")
        return dataloader
    except Exception as e:
        print(f"BSD100 Loading failed: {e}. Returning None.")
        import traceback
        traceback.print_exc()
        return None


In [ ]:
# ----------------------------------------------------------------------
# 4. MODEL DEFINITION - Real-ESRGAN Architecture
# ----------------------------------------------------------------------

class ResidualDenseBlock(nn.Module):
    """Dense Block as used in Real-ESRGAN."""
    def __init__(self, num_filters=64, growth_channel=32):
        super(ResidualDenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_filters, growth_channel, 3, padding=1)
        self.conv2 = nn.Conv2d(num_filters + growth_channel, growth_channel, 3, padding=1)
        self.conv3 = nn.Conv2d(num_filters + 2*growth_channel, growth_channel, 3, padding=1)
        self.conv4 = nn.Conv2d(num_filters + 3*growth_channel, growth_channel, 3, padding=1)
        self.conv5 = nn.Conv2d(num_filters + 4*growth_channel, num_filters, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2_input = torch.cat([x, x1], dim=1)
        x2 = self.lrelu(self.conv2(x2_input))
        x3_input = torch.cat([x, x1, x2], dim=1)
        x3 = self.lrelu(self.conv3(x3_input))
        x4_input = torch.cat([x, x1, x2, x3], dim=1)
        x4 = self.lrelu(self.conv4(x4_input))
        x5_input = torch.cat([x, x1, x2, x3, x4], dim=1)
        x5 = self.conv5(x5_input)
        return x5

class ResidualInResidualDenseBlock(nn.Module):
    def __init__(self, num_filters=64, growth_channel=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_filters, growth_channel)
        self.rdb2 = ResidualDenseBlock(num_filters, growth_channel)
        self.rdb3 = ResidualDenseBlock(num_filters, growth_channel)
    
    def forward(self, x):
        out = x + self.rdb1(x) * 0.2
        out = out + self.rdb2(out) * 0.2
        return out + self.rdb3(out) * 0.2

class RealESRGANGenerator(nn.Module):
    def __init__(self, scale=SCALE_FACTOR, num_rrdb=12):  # Keep same as baseline
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(3, 64, 3, padding=1)
        self.rrdb_blocks = nn.Sequential(*[ResidualInResidualDenseBlock(64, 32) for _ in range(num_rrdb)])
        self.conv_body = nn.Conv2d(64, 64, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        
        if scale == 4:
            self.conv_up1 = nn.Conv2d(64, 64*4, 3, padding=1)
            self.conv_up2 = nn.Conv2d(64, 64*4, 3, padding=1)
        elif scale == 2:
            self.conv_up1 = nn.Conv2d(64, 64*4, 3, padding=1)
        
        self.conv_hr = nn.Conv2d(64, 64, 3, padding=1)
        self.conv_last = nn.Conv2d(64, 3, 3, padding=1)
    
    def forward(self, x):
        feat = self.conv_first(x)
        global_res = feat
        feat = self.rrdb_blocks(feat)
        feat = self.conv_body(feat) + global_res
        
        if self.scale == 4:
            feat = F.pixel_shuffle(self.lrelu(self.conv_up1(feat)), 2)
            feat = F.pixel_shuffle(self.lrelu(self.conv_up2(feat)), 2)
        elif self.scale == 2:
            feat = F.pixel_shuffle(self.lrelu(self.conv_up1(feat)), 2)
        else:
            # Fallback for other scales
            feat = F.interpolate(feat, scale_factor=self.scale, mode='bilinear', align_corners=False)
        
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return torch.clamp(torch.tanh(out) * 0.58 + 0.5, 0, 1)

def create_esrgan_sr_model(scale=SCALE_FACTOR, lr_size=LR_SIZE, num_rrdb=None):
    """Defines a Real-ESRGAN based Super-Resolution model structure."""
    print("\nDefining PyTorch Real-ESRGAN Super-Resolution Model...")
    
    if num_rrdb is None:
        num_rrdb = NUM_RRDB_BLOCKS
    
    model = RealESRGANGenerator(scale=scale, num_rrdb=num_rrdb)
    model = model.to(device)
    
    print(f"Real-ESRGAN Model defined successfully with {num_rrdb} RRDB blocks.")
    return model




In [7]:
# ----------------------------------------------------------------------
# 5. DISCRIMINATOR (Simplified Patch Discriminator)
# ----------------------------------------------------------------------

class SimpleDiscriminator(nn.Module):
    """Simplified patch discriminator - much lighter than U-Net, uses ~90% less memory."""
    def __init__(self, input_channels=3):
        super(SimpleDiscriminator, self).__init__()
        
        # Simple sequential discriminator - no skip connections, no upsampling
        self.model = nn.Sequential(
            # Input: [B, 3, H, W]
            nn.Conv2d(input_channels, 32, 4, stride=2, padding=1),  # /2
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # /4
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # /8
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # /16
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Global average pooling and output
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(256, 1, 1),
        )
    
    def forward(self, x):
        x = self.model(x)
        # Output: [B, 1, 1, 1] -> [B, 1]
        return x.squeeze(-1).squeeze(-1)

def build_discriminator(input_channels=3):
    """Build simplified discriminator model."""
    model = SimpleDiscriminator(input_channels=input_channels)
    model = model.to(device)
    return model


In [8]:
# ----------------------------------------------------------------------
# 6. PERCEPTUAL LOSS (VGG19 feature extractor)
# ----------------------------------------------------------------------

class VGGFeatureExtractor(nn.Module):
    """VGG19 feature extractor for perceptual loss."""
    def __init__(self, layer_name='features.34'):  # block5_conv4 in PyTorch VGG
        super(VGGFeatureExtractor, self).__init__()
        vgg = vgg19(pretrained=True)
        # Extract features up to the specified layer
        features = list(vgg.features)
        self.features = nn.Sequential(*features[:35])  # Up to block5_conv4
        for param in self.features.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # Input is already normalized in perceptual_loss function
        return self.features(x)

def perceptual_loss(vgg_model, y_true, y_pred):
    """Compute perceptual loss using VGG features."""
    # Convert [0,1] to VGG expected range
    # VGG19 expects input normalized with ImageNet stats
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(y_true.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(y_true.device)
    
    y_true_normalized = (y_true - mean) / std
    y_pred_normalized = (y_pred - mean) / std
    
    f_true = vgg_model(y_true_normalized)
    f_pred = vgg_model(y_pred_normalized)
    return torch.mean(torch.abs(f_true - f_pred))

def build_vgg_feature_extractor(layer_name='features.34'):
    """Build VGG feature extractor."""
    model = VGGFeatureExtractor(layer_name=layer_name)
    model = model.to(device)
    model.eval()
    return model


In [ ]:
# ----------------------------------------------------------------------
# 7.5. VALIDATION EVALUATION FUNCTION
# ----------------------------------------------------------------------

def evaluate_validation(generator, val_loader, device, max_samples=50):
    """
    Evaluate model on validation set and return PSNR and SSIM.
    
    Args:
        generator: The generator model
        val_loader: Validation data loader
        device: Device to run evaluation on
        max_samples: Maximum number of samples to evaluate (for faster validation)
    
    Returns:
        Dictionary with 'psnr' and 'ssim' values
    """
    generator.eval()
    psnr_values = []
    ssim_values = []
    
    with torch.no_grad():
        for i, hr_batch in enumerate(val_loader):
            if i >= max_samples:
                break
            
            hr = hr_batch.to(device, non_blocking=True)
            
            # Generate LR using bicubic downsampling
            lr = generate_bicubic_lr(hr, scale=SCALE_FACTOR)
            
            # Generate SR
            sr = generator(lr)
            
            # Calculate metrics
            batch_psnr = psnr(hr, sr).item()
            batch_ssim = ssim(hr, sr).item()
            
            psnr_values.append(batch_psnr)
            ssim_values.append(batch_ssim)
    
    generator.train()  # Set back to training mode
    
    return {
        'psnr': np.mean(psnr_values) if len(psnr_values) > 0 else 0.0,
        'ssim': np.mean(ssim_values) if len(ssim_values) > 0 else 0.0
    }


In [ ]:
# ----------------------------------------------------------------------
# 7.6. PLOTTING FUNCTIONS FOR TRAINING AND VALIDATION METRICS
# ----------------------------------------------------------------------

def plot_psnr_ssim_simple(epochs, train_psnr, train_ssim, val_psnr=None, val_ssim=None, save_path='psnr_ssim_plot.png'):
    """
    Simple plot of PSNR and SSIM with optional validation metrics.
    
    Args:
        epochs: List of epoch numbers
        train_psnr: List of training PSNR values
        train_ssim: List of training SSIM values
        val_psnr: Optional list of validation PSNR values
        val_ssim: Optional list of validation SSIM values
        save_path: Path to save the plot
    """
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # Plot Training PSNR on left y-axis
    color1 = 'tab:blue'
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('PSNR (dB)', color=color1, fontsize=12)
    line1 = ax1.plot(epochs, train_psnr, 'o-', color=color1, linewidth=2, markersize=5, label='Train PSNR', alpha=0.8)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, alpha=0.3)
    
    # Plot Validation PSNR if available
    if val_psnr is not None:
        line1_val = ax1.plot(epochs, val_psnr, 's--', color='lightblue', linewidth=2, markersize=5, label='Val PSNR', alpha=0.8)
        line1 = line1 + line1_val
    
    # Plot Training SSIM on right y-axis
    ax2 = ax1.twinx()
    color2 = 'tab:green'
    ax2.set_ylabel('SSIM', color=color2, fontsize=12)
    line2 = ax2.plot(epochs, train_ssim, '^-', color=color2, linewidth=2, markersize=5, label='Train SSIM', alpha=0.8)
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # Plot Validation SSIM if available
    if val_ssim is not None:
        line2_val = ax2.plot(epochs, val_ssim, 'd--', color='lightgreen', linewidth=2, markersize=5, label='Val SSIM', alpha=0.8)
        line2 = line2 + line2_val
    
    # Combine legends
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='best', fontsize=10)
    
    title = 'Training and Validation PSNR & SSIM' if (val_psnr is not None) else 'Training PSNR & SSIM'
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Plot saved to: {save_path}")


In [ ]:
# ----------------------------------------------------------------------
# 7. GAN TRAINING LOOP
# ----------------------------------------------------------------------

def compute_generator_losses(generator, discriminator, vgg, lr, hr, sr, 
                             lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005, 
                             use_mse=False, use_charbonnier=False):
    """Compute generator losses: pixel, perceptual, and adversarial."""
    # Pixel loss: Charbonnier (best for very high PSNR), MSE, or L1
    if use_charbonnier:
        # Charbonnier loss (smooth L1) - often achieves best PSNR
        epsilon = 1e-6
        diff = hr - sr
        pixel_loss = torch.mean(torch.sqrt(diff ** 2 + epsilon))
    elif use_mse:
        pixel_loss = torch.mean((hr - sr) ** 2)  # MSE loss - better for PSNR
    else:
        pixel_loss = torch.mean(torch.abs(hr - sr))  # L1 loss
    
    # Perceptual loss (only if vgg is not None and lambda_percep > 0)
    if vgg is not None and lambda_percep > 0:
        percep_loss = perceptual_loss(vgg, hr, sr)
    else:
        percep_loss = torch.tensor(0.0, device=hr.device)
    
    # RaGAN generator loss (only if discriminator is not None and lambda_adv > 0)
    if discriminator is not None and lambda_adv > 0:
        d_real = discriminator(hr)
        d_fake = discriminator(sr)
        mean_real = torch.mean(d_real)
        
        # Generator wants D_fake - E[D_real] to be classified as real
        g_adv_loss = F.binary_cross_entropy_with_logits(d_fake - mean_real, torch.ones_like(d_fake))
    else:
        g_adv_loss = torch.tensor(0.0, device=hr.device)
    
    
    total_loss = lambda_pixel * pixel_loss + lambda_percep * percep_loss + lambda_adv * g_adv_loss
    
    return total_loss, pixel_loss, percep_loss, g_adv_loss

def compute_discriminator_loss(discriminator, hr, sr):
    """Compute RaGAN discriminator loss."""
    d_real = discriminator(hr)
    d_fake = discriminator(sr)
    mean_real = torch.mean(d_real)
    mean_fake = torch.mean(d_fake)
    
    # Discriminator: classify D_real - E[D_fake] as real, D_fake - E[D_real] as fake
    real_loss = F.binary_cross_entropy_with_logits(d_real - mean_fake, torch.ones_like(d_real))
    fake_loss = F.binary_cross_entropy_with_logits(d_fake - mean_real, torch.zeros_like(d_fake))
    
    return real_loss + fake_loss

def train_one_epoch(generator, discriminator, vgg, train_loader, 
                    g_optimizer, d_optimizer, epoch, device, 
                    lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005, 
                    use_mse=False, use_charbonnier=False, use_mixed_precision=False):
    """Train for one epoch."""
    generator.train()
    if discriminator is not None:
        discriminator.train()
    
    # Mixed precision training setup (FP16 - saves ~40-50% memory)
    scaler = torch.cuda.amp.GradScaler() if use_mixed_precision and torch.cuda.is_available() else None
    
    g_losses = []
    d_losses = []
    psnr_values = []
    ssim_values = []
    
    for batch_idx, hr_batch in enumerate(train_loader):
        # Move HR to device with non-blocking transfer
        hr = hr_batch.to(device, non_blocking=True)
        
        # Generate LR on GPU (much faster than CPU, better GPU utilization)
        # This moves the bottleneck from CPU data loading to GPU compute
        with torch.no_grad():
            lr = generate_bicubic_lr(hr, scale=SCALE_FACTOR)
        
        # 1) Update Discriminator (only if adversarial loss is enabled)
        if lambda_adv > 0 and d_optimizer is not None:
            # Only update discriminator every 2 batches for more stable training
            if batch_idx % 2 == 0:  # Update D every 2 batches, G every batch
                d_optimizer.zero_grad()
                
                if scaler is not None:
                    # Mixed precision for discriminator
                    with torch.no_grad(), torch.cuda.amp.autocast():
                        sr_d = generator(lr)
                    
                    with torch.cuda.amp.autocast():
                        d_loss = compute_discriminator_loss(discriminator, hr, sr_d.detach())
                    
                    scaler.scale(d_loss).backward()
                    scaler.unscale_(d_optimizer)
                    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRADIENT_CLIP_VALUE)
                    scaler.step(d_optimizer)
                    scaler.update()
                else:
                    # Full precision
                    with torch.no_grad():
                        sr_d = generator(lr)
                    
                    d_loss = compute_discriminator_loss(discriminator, hr, sr_d.detach())
                    d_loss.backward()
                    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRADIENT_CLIP_VALUE)
                    d_optimizer.step()
                
                # Store d_loss value before clearing
                d_loss_val = d_loss.item()
                d_losses.append(d_loss_val)
                
                # Clear intermediate variables immediately
                del d_loss, sr_d
            else:
                # Use previous d_loss for logging
                d_loss_val = d_losses[-1] if len(d_losses) > 0 else 0.0
        else:
            # Discriminator disabled - no adversarial loss
            d_loss_val = 0.0
        
        # Clear cache more frequently to reduce memory usage
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # 2) Update Generator (update every batch for better convergence)
        g_optimizer.zero_grad()
        
        if scaler is not None:
            # Mixed precision for generator
            with torch.cuda.amp.autocast():
                sr = generator(lr)
                g_total, l_pix, l_perc, l_adv = compute_generator_losses(
                    generator, discriminator, vgg, lr, hr, sr,
                    lambda_pixel, lambda_percep, lambda_adv, 
                    use_mse=use_mse, use_charbonnier=use_charbonnier
                )
            
            scaler.scale(g_total).backward()
            scaler.unscale_(g_optimizer)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), GRADIENT_CLIP_VALUE)
            scaler.step(g_optimizer)
            scaler.update()
        else:
            # Full precision
            sr = generator(lr)
            g_total, l_pix, l_perc, l_adv = compute_generator_losses(
                generator, discriminator, vgg, lr, hr, sr,
                lambda_pixel, lambda_percep, lambda_adv, 
                use_mse=use_mse, use_charbonnier=use_charbonnier
            )
            g_total.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), GRADIENT_CLIP_VALUE)
            g_optimizer.step()
        
        # Metrics (compute before clearing variables)
        with torch.no_grad():
            batch_psnr = psnr(hr, sr).item()
            batch_ssim = ssim(hr, sr).item()
            psnr_values.append(batch_psnr)
            ssim_values.append(batch_ssim)
        
        g_losses.append(g_total.item())
        
        # Clear variables to free memory
        del sr, g_total, l_pix, l_perc, l_adv
        
        # Periodic GPU cache clearing for memory management (more frequent = lower memory)
        if CLEAR_CACHE_FREQUENCY > 0 and (batch_idx + 1) % CLEAR_CACHE_FREQUENCY == 0:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()  # Ensure all operations complete before clearing
        
        if (batch_idx + 1) % 10 == 0:
            current_lr = g_optimizer.param_groups[0]['lr']
            print(f'Epoch {epoch}, Batch {batch_idx+1}/{len(train_loader)}, '
                  f'G_Loss: {g_losses[-1]:.4f}, D_Loss: {d_loss_val:.4f}, '
                  f'PSNR: {batch_psnr:.2f}, SSIM: {batch_ssim:.4f}, LR: {current_lr:.2e}')
    
    # Average metrics (handle case where d_losses might be shorter)
    avg_d_loss = np.mean(d_losses) if len(d_losses) > 0 else 0.0
    return {
        'g_loss': np.mean(g_losses),
        'd_loss': avg_d_loss,
        'psnr': np.mean(psnr_values),
        'ssim': np.mean(ssim_values)
    }


In [ ]:
# ----------------------------------------------------------------------
# 8. EVALUATION COMPONENTS
# ----------------------------------------------------------------------

def predict_super_resolution(model, lr_image):
    """Generate super-resolved image from LR input."""
    model.eval()
    with torch.no_grad():
        lr_batched = lr_image.unsqueeze(0).to(device)
        sr_float = model(lr_batched)
        sr_float = sr_float.squeeze(0)
        sr_image_uint8 = torch.clamp(sr_float * 255.0, 0, 255).byte()
    return sr_image_uint8.cpu(), sr_float.cpu()

def plot_lr_sr_hr(lr_image, sr_image, hr_image, psnr, ssim, index, scale=SCALE_FACTOR):
    """
    Displays the LR Input, SR Output, and HR Ground Truth comparison by saving the plot to a file.
    Includes calculated metrics for the current sample.
    """
    # Determine the display size (HR dimensions)
    hr_h, hr_w = hr_image.shape[1], hr_image.shape[2]
    
    # Create the figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Resize LR for visualization only (using nearest neighbor for clarity)
    # Ensure lr_image is in (C, H, W) format
    if len(lr_image.shape) == 3:
        # Already in (C, H, W) format, add batch dimension for interpolation
        lr_batched = lr_image.unsqueeze(0)  # (1, C, H, W)
    else:
        # Already has batch dimension or different format
        lr_batched = lr_image if len(lr_image.shape) == 4 else lr_image.unsqueeze(0)
    
    # Interpolate to match HR dimensions
    lr_display = F.interpolate(lr_batched, size=(hr_h, hr_w), mode='nearest').squeeze(0)
    lr_display = (lr_display.permute(1, 2, 0) * 255).byte().cpu().numpy()
    
    # Handle SR image dimensions - crop to match HR for display
    sr_h, sr_w = sr_image.shape[1], sr_image.shape[2]
    min_h, min_w = min(hr_h, sr_h), min(hr_w, sr_w)
    sr_cropped = sr_image[:, :min_h, :min_w]
    hr_cropped = hr_image[:, :min_h, :min_w]
    
    # Convert HR and SR (uint8) for display
    hr_display = (hr_cropped.permute(1, 2, 0) * 255).byte().cpu().numpy()
    sr_display = sr_cropped.permute(1, 2, 0).byte().cpu().numpy()
    
    axes[0].imshow(lr_display)
    axes[0].set_title(f"Low Resolution Input (x{scale} Bicubic)", fontsize=10)
    axes[0].axis("off")
    
    axes[1].imshow(sr_display.astype(np.uint8))
    axes[1].set_title(f"SR Output (PSNR: {psnr:.2f} dB, SSIM: {ssim:.4f})", fontsize=10)
    axes[1].axis("off")
    
    axes[2].imshow(hr_display.astype(np.uint8))
    axes[2].set_title(f"High Resolution Ground Truth", fontsize=10)
    axes[2].axis("off")
    
    plt.suptitle(f"BSD100 Test Sample {index+1}", fontsize=12)
    plt.tight_layout()
    
    # Save the plot to a file in the improved version directory
    filepath = os.path.join(OUTPUT_DIR, f"bsd100_test_comparison_sample_{index+1}.png")
    plt.savefig(filepath)
    plt.close(fig)  # Close the figure to free up memory
    print(f"Plot saved to {filepath}")

def run_test_evaluation_bsd100(model, test_loader, num_samples=5):
    """
    Runs evaluation on the BSD100 dataset, calculating metrics and saving plots.
    """
    print(f"\n--- Starting BSD100 Test Evaluation ({num_samples} Samples) ---")
    
    model.eval()
    total_psnr = 0.0
    total_ssim = 0.0
    count = 0
    
    # Iterate over the first few samples for visual plotting
    with torch.no_grad():
        for i, (lr_batch, hr_batch) in enumerate(test_loader):
            if i >= num_samples:
                break
            
            # Extract the single image from the batch
            lr_img_norm = lr_batch[0]  # Normalized LR [0, 1]
            hr_img_norm = hr_batch[0]  # Normalized HR [0, 1]
            
            # Upscale the image
            sr_img_uint8, sr_img_norm = predict_super_resolution(model, lr_img_norm)
            
            # Ensure HR and SR have matching dimensions for metric calculation
            # Align to the minimum size to avoid dimension mismatches
            hr_h, hr_w = hr_img_norm.shape[1], hr_img_norm.shape[2]
            sr_h, sr_w = sr_img_norm.shape[1], sr_img_norm.shape[2]
            
            # Use the minimum dimensions to crop both tensors
            min_h = min(hr_h, sr_h)
            min_w = min(hr_w, sr_w)
            
            # Crop both tensors to match
            hr_cropped = hr_img_norm[:, :min_h, :min_w].unsqueeze(0).to(device)
            sr_cropped = sr_img_norm[:, :min_h, :min_w].unsqueeze(0).to(device)
            
            # Calculate metrics for this sample
            current_psnr = psnr(hr_cropped, sr_cropped).item()
            # SSIM returns a single value if both inputs have the same shape
            current_ssim = ssim(hr_cropped, sr_cropped).item()
            
            # Accumulate metrics
            total_psnr += current_psnr
            total_ssim += current_ssim
            count += 1
            
            # Plot the LR, SR, and HR results, including metrics
            plot_lr_sr_hr(lr_img_norm, sr_img_uint8, hr_img_norm, current_psnr, current_ssim, i)
    
    # Calculate and print final mean metrics if data was processed
    if count > 0:
        mean_psnr = total_psnr / count
        mean_ssim = total_ssim / count
        print(f"\n--- RESULTS ON BSD100 TEST SET (First {count} Samples) ---")
        print(f"Mean PSNR: {mean_psnr:.4f} dB")
        print(f"Mean SSIM: {mean_ssim:.4f}")
    else:
        print("No BSD100 test samples were available for evaluation.")
    
    print("--- BSD100 Test Evaluation Complete. ---")


In [ ]:
# ----------------------------------------------------------------------
# 9. MAIN TRAINING AND EVALUATION SCRIPT
# ----------------------------------------------------------------------

if __name__ == '__main__':
    # 1. Initialize datasets
    train_loader = load_or_simulate_dataset(
        num_samples=TRAIN_SAMPLES,
        batch_size=BATCH_SIZE,
        lr_shape=(LR_SIZE, LR_SIZE, 3),
        hr_shape=(HR_SIZE, HR_SIZE, 3)
    )
    
    # Create validation dataset loader (for per-epoch validation)
    try:
        print("\nCreating validation dataset loader...")
        val_dataset = DIV2KDataset(num_samples=100, lr_size=LR_SIZE, hr_size=HR_SIZE, data_path=VALID_LR_PATH)
        val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)
        print(f"Validation dataset loaded: {len(val_dataset)} samples")
        USE_VALIDATION = True
    except Exception as e:
        print(f"Warning: Could not create validation loader: {e}")
        print("Continuing without validation evaluation...")
        val_loader = None
        USE_VALIDATION = False
    
    test_loader = load_test_dataset()
    
    # 2. Build models (same structure as baseline)
    generator = create_esrgan_sr_model(scale=SCALE_FACTOR, num_rrdb=NUM_RRDB_BLOCKS)
    
    # Only create discriminator if adversarial loss is enabled
    if LAMBDA_ADV > 0:
        discriminator = build_discriminator(input_channels=3)
    else:
        discriminator = None  # Not needed when adversarial loss is disabled
    
    # Only create VGG if perceptual loss is enabled
    if LAMBDA_PERCEP > 0:
        vgg = build_vgg_feature_extractor('features.34')
    else:
        vgg = None  # Not needed when perceptual loss is disabled
    
    # 3. Optimizers with learning rate schedule (optimized for PSNR)
    # Lower learning rate for more stable training and higher PSNR
    initial_lr = 5e-5  # Further reduced for stable convergence (was 1e-4)
    g_optimizer = optim.Adam(generator.parameters(), lr=initial_lr, betas=(0.9, 0.99))
    
    # Only create discriminator optimizer if adversarial loss is enabled
    if LAMBDA_ADV > 0:
        d_optimizer = optim.Adam(discriminator.parameters(), lr=initial_lr, betas=(0.9, 0.99))
        d_scheduler = None  # Will set later if needed
    else:
        d_optimizer = None
        print("Adversarial loss disabled - training without discriminator for maximum PSNR")
    
    # Learning rate scheduler - more gradual decay for stable PSNR improvement
    batches_per_epoch = len(train_loader)
    milestone_epochs = [10, 20]  # Decay earlier for fine-tuning
    milestone_batches = [ep * batches_per_epoch for ep in milestone_epochs]
    g_scheduler = optim.lr_scheduler.MultiStepLR(g_optimizer, milestones=milestone_batches, gamma=0.5)
    
    if d_optimizer is not None:
        d_scheduler = optim.lr_scheduler.MultiStepLR(d_optimizer, milestones=milestone_batches, gamma=0.5)
    else:
        d_scheduler = None
    
    print(f"Learning rate: {initial_lr}, Scheduler milestones (epochs): {milestone_epochs}")
    
    # Option to load pretrained model for fine-tuning
    PRETRAINED_MODEL_PATH = 'best-esrgan-generator-pytorch.pth'  # Original model
    LOAD_PRETRAINED = False  # Set to True to continue training from original model
    
    if LOAD_PRETRAINED and os.path.exists(PRETRAINED_MODEL_PATH):
        print(f"Loading pretrained model from {PRETRAINED_MODEL_PATH}...")
        try:
            generator.load_state_dict(torch.load(PRETRAINED_MODEL_PATH, map_location=device))
            print("Pretrained model loaded successfully. Continuing training...")
        except Exception as e:
            print(f"Warning: Could not load pretrained model: {e}")
            print("Starting training from scratch...")
    
    print("\n" * 2)
    
        # 4. Training (aligned with baseline structure)
    print(f"--- Starting Real-ESRGAN GAN Training for PSNR Optimization ({EPOCHS} Epochs) ---")
    print(f"Loss Configuration (PSNR-maximization):")
    print(f"  Pixel Loss Weight: {LAMBDA_PIXEL}, Perceptual: {LAMBDA_PERCEP}, Adversarial: {LAMBDA_ADV}")
    print(f"  Using {'Charbonnier' if USE_CHARBONNIER else 'MSE' if USE_MSE_LOSS else 'L1'} pixel loss")
    if LAMBDA_ADV == 0:
        print(f"  ⚠️  Adversarial loss DISABLED - pure pixel/perceptual training for maximum PSNR")
    print(f"  Gradient clipping: {GRADIENT_CLIP_VALUE}")
    print(f"  Mixed Precision (FP16): {USE_MIXED_PRECISION} (saves ~40-50% memory)")
    print(f"  Cache clearing: every {CLEAR_CACHE_FREQUENCY} batches")
    print("-" * 60)
    
    best_psnr = 0.0
    best_loss = float('inf')
    best_val_psnr = 0.0
    
    # Lists to store metrics for plotting
    all_epochs = []
    train_psnr_list = []
    train_ssim_list = []
    val_psnr_list = []
    val_ssim_list = []
    
    for epoch in range(1, EPOCHS + 1):
        metrics = train_one_epoch(
            generator, discriminator, vgg, train_loader,
            g_optimizer, d_optimizer, epoch, device,
            lambda_pixel=LAMBDA_PIXEL, 
            lambda_percep=LAMBDA_PERCEP, 
            lambda_adv=LAMBDA_ADV,
            use_mse=USE_MSE_LOSS and not USE_CHARBONNIER,
            use_charbonnier=USE_CHARBONNIER,
            use_mixed_precision=USE_MIXED_PRECISION
        )
        
        # Store training metrics
        all_epochs.append(epoch)
        train_psnr_list.append(metrics['psnr'])
        train_ssim_list.append(metrics['ssim'])
        
        # Run validation evaluation
        val_metrics = None
        if USE_VALIDATION and val_loader is not None:
            val_metrics = evaluate_validation(generator, val_loader, device, max_samples=50)
            val_psnr_list.append(val_metrics['psnr'])
            val_ssim_list.append(val_metrics['ssim'])
        else:
            val_psnr_list.append(None)
            val_ssim_list.append(None)
        
        print(f"\nEpoch {epoch}/{EPOCHS} Summary:")
        print(f"  G_Loss: {metrics['g_loss']:.4f}, D_Loss: {metrics['d_loss']:.4f}")
        print(f"  Train PSNR: {metrics['psnr']:.2f} dB, Train SSIM: {metrics['ssim']:.4f}")
        if val_metrics:
            print(f"  Val PSNR: {val_metrics['psnr']:.2f} dB, Val SSIM: {val_metrics['ssim']:.4f}")
        
        # Save best model based on validation PSNR (if available) or training PSNR
        if val_metrics and val_metrics['psnr'] > best_val_psnr:
            best_val_psnr = val_metrics['psnr']
            model_path = os.path.join(OUTPUT_DIR, 'best-esrgan-generator-psnr-improved.pth')
            torch.save(generator.state_dict(), model_path)
            print(f"  ✓ Saved best model (Val PSNR: {best_val_psnr:.2f} dB) to {model_path}")
        elif metrics['psnr'] > best_psnr:
            best_psnr = metrics['psnr']
            model_path = os.path.join(OUTPUT_DIR, 'best-esrgan-generator-psnr-improved.pth')
            torch.save(generator.state_dict(), model_path)
            print(f"  ✓ Saved best model (Train PSNR: {best_psnr:.2f} dB) to {model_path}")
        
        # Also save based on loss (backup)
        if metrics['g_loss'] < best_loss:
            best_loss = metrics['g_loss']
            if (val_metrics and val_metrics['psnr'] <= best_val_psnr) or (not val_metrics and metrics['psnr'] <= best_psnr):
                model_path = os.path.join(OUTPUT_DIR, 'best-esrgan-generator-loss-improved.pth')
                torch.save(generator.state_dict(), model_path)
        
        # Plot metrics every 5 epochs and at the end
        if epoch % 5 == 0 or epoch == EPOCHS:
            val_psnr_plot = val_psnr_list if USE_VALIDATION else None
            val_ssim_plot = val_ssim_list if USE_VALIDATION else None
            plot_path = os.path.join(OUTPUT_DIR, 'training_metrics.png')
            plot_psnr_ssim_simple(all_epochs, train_psnr_list, train_ssim_list, 
                                 val_psnr=val_psnr_plot, val_ssim=val_ssim_plot, 
                                 save_path=plot_path)
            print(f"  Metrics plot updated at epoch {epoch}")
        
        g_scheduler.step()
        if d_scheduler is not None:
            d_scheduler.step()
    
    # Final plot and summary
    print(f"\n--- Training Complete ---")
    print(f"Final Training Metrics:")
    print(f"  Best Train PSNR: {max(train_psnr_list):.2f} dB | Best Train SSIM: {max(train_ssim_list):.4f}")
    if USE_VALIDATION and len([v for v in val_psnr_list if v is not None]) > 0:
        valid_val_psnr = [v for v in val_psnr_list if v is not None]
        valid_val_ssim = [v for v in val_ssim_list if v is not None]
        print(f"  Best Val PSNR: {max(valid_val_psnr):.2f} dB | Best Val SSIM: {max(valid_val_ssim):.4f}")
    
    # Final plot
    val_psnr_final = val_psnr_list if USE_VALIDATION else None
    val_ssim_final = val_ssim_list if USE_VALIDATION else None
    final_plot_path = os.path.join(OUTPUT_DIR, 'training_metrics_final.png')
    plot_psnr_ssim_simple(all_epochs, train_psnr_list, train_ssim_list, 
                         val_psnr=val_psnr_final, val_ssim=val_ssim_final, 
                         save_path=final_plot_path)
    print("Final training metrics plot saved.")
    



Attempting to load real DIV2K dataset...
Attempting to load DIV2K dataset from D:\Documents\GitHub\AI6132-Group-Project\datasets\DIV2K_train_LR_bicubic\X4...
Found 800 images in dataset folder.
Successfully loaded 800 DIV2K images.
TFDS DIV2K dataset processing configured with BATCH_SIZE=8.

Attempting to load BSD100 (B100) Test Set via TFDS...

Attempting to load BSD100 Test Set from D:\Documents\GitHub\AI6132-Group-Project\datasets\DIV2K_valid_LR_bicubic\X4...
Found 100 images in validation dataset folder.
Successfully loaded 100 BSD100 validation images.
BSD100 Test Set configured for evaluation (LR generated via Bicubic downsampling).

Defining PyTorch Real-ESRGAN Super-Resolution Model...
Real-ESRGAN Model defined successfully.


C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)





--- Starting Real-ESRGAN GAN Training (30 Epochs) ---
Epoch 1, Batch 10/100, G_Loss: 0.4828, D_Loss: 0.7942, PSNR: 11.08, SSIM: -0.0329
Epoch 1, Batch 20/100, G_Loss: 0.4418, D_Loss: 0.6086, PSNR: 11.71, SSIM: 0.0222
Epoch 1, Batch 30/100, G_Loss: 0.4717, D_Loss: 0.8458, PSNR: 12.85, SSIM: 0.5091
Epoch 1, Batch 40/100, G_Loss: 0.4097, D_Loss: 0.6272, PSNR: 15.98, SSIM: 0.6983
Epoch 1, Batch 50/100, G_Loss: 0.3951, D_Loss: 0.6196, PSNR: 16.19, SSIM: 0.7362
Epoch 1, Batch 60/100, G_Loss: 0.3306, D_Loss: 0.4718, PSNR: 18.29, SSIM: 0.8414
Epoch 1, Batch 70/100, G_Loss: 0.3257, D_Loss: 0.3359, PSNR: 16.19, SSIM: 0.7550
Epoch 1, Batch 80/100, G_Loss: 0.3557, D_Loss: 0.3976, PSNR: 15.41, SSIM: 0.7279
Epoch 1, Batch 90/100, G_Loss: 0.4077, D_Loss: 0.3300, PSNR: 15.27, SSIM: 0.6602
Epoch 1, Batch 100/100, G_Loss: 0.3538, D_Loss: 0.3013, PSNR: 16.37, SSIM: 0.7595

Epoch 1/30 Summary:
  G_Loss: 0.4235, D_Loss: 0.5718
  PSNR: 14.86, SSIM: 0.5427
  Saved best model (G_Loss: 0.4235)
Epoch 2, Batc

Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py", line 1284, in _try_get_data
    data = self._data_queue.get(timeout=timeout)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\queue.py", line 209, in get
    raise Empty
_queue.Empty

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_16328\3958063569.py", line 60, in <module>
    run_test_evaluation_bsd100(generator, test_loader, num_samples=5)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_16328\1523755489.py", line 62, in run_test_evaluation_bsd100
    for i, (lr_batch, hr_batch) in enumerate(test_loader):
                                   ~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Pr

SystemExit: 1

C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [14]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asilva1691/bsd100")

print("Path to dataset files:", path)

C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 60.8M/60.8M [00:04<00:00, 15.6MB/s]


Extracting files...
Path to dataset files: C:\Users\Administrator\.cache\kagglehub\datasets\asilva1691\bsd100\versions\1


In [ ]:
# ----------------------------------------------------------------------
# SIMPLE PSNR & SSIM PLOTTING (Run after training to visualize metrics)
# ----------------------------------------------------------------------
# This cell can be used to manually plot metrics if you have the data

# If you want to manually plot after training, uncomment and fill in your metrics:
# all_epochs = [1, 2, 3, ...]
# train_psnr = [14.86, 18.88, ...]
# train_ssim = [0.5427, 0.8646, ...]
# val_psnr = [14.50, 18.20, ...]  # Optional
# val_ssim = [0.5000, 0.8500, ...]  # Optional

# plot_psnr_ssim_simple(all_epochs, train_psnr, train_ssim, 
#                      val_psnr=val_psnr, val_ssim=val_ssim,
#                      save_path=os.path.join(OUTPUT_DIR, 'manual_plot.png'))

print("Plotting function is ready. Metrics are automatically plotted during training.")


In [ ]:
# ----------------------------------------------------------------------
# EVALUATION: Load Model and Run BSD100 Test Evaluation
# ----------------------------------------------------------------------

print("--- Training Complete. Starting Evaluation. ---")

# Check if generator is already defined, if not create it
try:
    # Try to use existing generator
    _ = generator
    print("Using existing generator model.")
except NameError:
    # Create new generator if it doesn't exist
    print("Generator not found. Creating new model...")
    generator = create_esrgan_sr_model(scale=SCALE_FACTOR)
    
    # Try to load the best PSNR model first, then fallback to other models
    model_paths = [
        os.path.join(OUTPUT_DIR, 'best-esrgan-generator-psnr-improved.pth'),  # PSNR-optimized model
        os.path.join(OUTPUT_DIR, 'best-esrgan-generator-loss-improved.pth'),  # Loss-optimized model
        'best-esrgan-generator-pytorch.pth'  # Original baseline model (in root)
    ]
    
    model_loaded = False
    for model_path in model_paths:
        try:
            generator.load_state_dict(torch.load(model_path, map_location=device))
            print(f"Successfully loaded model: {model_path}")
            model_loaded = True
            break
        except FileNotFoundError:
            continue
        except Exception as e:
            print(f"Warning: Could not load {model_path}: {e}")
            continue
    
    if not model_loaded:
        print("Warning: No model files found. Using current model state.")

# Load test dataset
test_loader = load_test_dataset()

try:
    # 4. Evaluation using BSD100
    if test_loader:
        # Run the formal test evaluation on BSD100
        run_test_evaluation_bsd100(generator, test_loader, num_samples=5)
    else:
        # Fallback to ad-hoc visual evaluation on training data if BSD100 failed to load
        print("\nWARNING: Could not load BSD100. Running ad-hoc visual check on training data.")
        # We'll keep a simplified version of the old function name for this fallback
        def run_ad_hoc_evaluation_fallback(model, loader, num_samples=8):
            print(f"\n--- Starting Ad-Hoc Visual Evaluation ({num_samples} Samples, Model: Real-ESRGAN) ---")
            model.eval()
            with torch.no_grad():
                for i, (lr_batch, _) in enumerate(loader):
                    if i >= num_samples:
                        break
                    lowres_img = lr_batch[0]
                    # Note: predict_super_resolution now takes normalized input
                    sr_img_uint8, _ = predict_super_resolution(model, lowres_img)
                    
                    # Create a simple LR vs SR plot
                    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                    
                    # Resizing LR input to match SR output size for visualization
                    sr_h, sr_w = sr_img_uint8.shape[1], sr_img_uint8.shape[2]
                    lr_display = F.interpolate(lowres_img.unsqueeze(0), 
                                               size=(sr_h, sr_w),
                                               mode='nearest').squeeze(0)
                    lr_display = (lr_display.permute(1, 2, 0) * 255).byte().cpu().numpy()
                    
                    axes[0].imshow(lr_display)
                    axes[0].axis("off")
                    axes[0].set_title("Low Resolution Input", fontsize=10)
                    
                    sr_display = sr_img_uint8.permute(1, 2, 0).byte().cpu().numpy()
                    axes[1].imshow(sr_display)
                    axes[1].axis("off")
                    axes[1].set_title("Real-ESRGAN Super-Resolution Output", fontsize=10)
                    
                    plt.tight_layout()
                    filepath = os.path.join(OUTPUT_DIR, f"esrgan_sr_comparison_sample_{i+1}_fallback.png")
                    plt.savefig(filepath)
                    plt.close(fig)
                    print(f"Fallback plot saved to {filepath}")
            print("--- Ad-Hoc Visual Evaluation Complete. ---")
        
        run_ad_hoc_evaluation_fallback(generator, train_loader, num_samples=8)
    
    print("\nModel Evaluation successfully completed.")
    
except Exception as e:
    print(f"\nFATAL ERROR: The script failed unexpectedly during evaluation.")
    print(f"Error detail: {e}")
    import traceback
    traceback.print_exc()

--- Training Complete. Starting Evaluation. ---

--- Starting BSD100 Test Evaluation with Real-ESRGAN (5 Samples) ---

FATAL ERROR: The script failed unexpectedly during evaluation.
Error detail: DataLoader worker (pid(s) 24388) exited unexpectedly


Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py", line 1284, in _try_get_data
    data = self._data_queue.get(timeout=timeout)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\queue.py", line 209, in get
    raise Empty
_queue.Empty

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_16328\3620043984.py", line 6, in <module>
    run_test_evaluation_bsd100(generator, test_loader, num_samples=5)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_16328\1523755489.py", line 62, in run_test_evaluation_bsd100
    for i, (lr_batch, hr_batch) in enumerate(test_loader):
                                   ~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Pro

SystemExit: 1

C:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
